## 1. Setup — Drive e clone della repo MSAE

Monta Google Drive e clona la repo ufficiale [WolodjaZ/MSAE](https://github.com/WolodjaZ/MSAE).

**Nessun `pip install` necessario**: il codice MSAE usa solo API standard di `torch`, `numpy`, `scipy`, `scikit-learn`, `tqdm` — tutto già preinstallato su Colab in versioni più recenti compatibili.



In [ ]:
from google.colab import drive
import pathlib

drive.mount('/content/drive')

# Clone della repo MSAE
!git clone https://github.com/WolodjaZ/MSAE.git /content/MSAE

%cd /content/MSAE


## 1ter. Sostituzione dei file MSAE modificati

Il `git clone` porta la repo originale. Sovrascriviamo `train.py` e `utils.py` con le versioni modificate che aggiungono:
- early stopping con patience=20 (`train.py`)
- argomento `-dv / --dataset_val` (`train.py`)
- supporto a parametri di normalizzazione esterni in `SAEDataset` (`utils.py`)

**Prerequisito**: caricare `MSAE/train.py` e `MSAE/utils.py` dal PC su Drive in `MyDrive/tesi/msae_patches/`.

In [ ]:
import shutil
from pathlib import Path

PATCHES = Path("/content/drive/MyDrive/tesi/msae_patches")

for fname in ["train.py", "utils.py"]:
    src = PATCHES / fname
    dst = Path("/content/MSAE") / fname
    if not src.exists():
        raise FileNotFoundError(
            f"File non trovato su Drive: {src}\n"
            f"Carica MSAE/{fname} dal tuo PC in MyDrive/tesi/msae_patches/"
        )
    shutil.copy(src, dst)
    print(f"Sostituito: {fname}")

## 1bis. Patch dei default della repo MSAE

Due default in `config.py` non corrispondono al setup del paper sul nostro subset:

- **`max_nesting=256`** → la repo costruisce la K-list Matryoshka raddoppiando da `nesting_list=64` fino a `max_nesting`. Con `256` ottieni solo `[64, 128, 256]` (3 livelli annidati su 6144 latenti). Il paper usa una K-list che copre tutta la latente, quindi serve `max_nesting=6144` per ottenere `[64, 128, 256, 512, 1024, 2048, 4096, 6144]` (8 livelli). 

- **`batch_size=4096`** → su 26 325 sample del subset darebbe solo ~6 step di gradiente per epoca. Portandolo a `256` otteniamo ~103 step/epoca → ~10k step totali su 100 epoche, ordine di grandezza dei ~21k step del paper su CC3M (708 step/epoca × 30 epoche).

`train.py` non espone questi due parametri come CLI flag, quindi li patchiamo direttamente in `config.py` con una regex prima del lancio. `--epochs` invece è esposto (`-e/--epochs`, default 100) ed è già impostato a 100 nella cella di training.

In [ ]:
import re
import subprocess
from pathlib import Path

cfg_path = Path("/content/MSAE/config.py")
src = cfg_path.read_text()

# max_nesting: 256 -> 6144 (K-list completa: [64, 128, 256, 512, 1024, 2048, 4096, 6144])
src_new = re.sub(#con una regex cerco dove il max nesting è impostato a 256 e lo imposto a 6144(768 x 8)
    r"max_nesting:\s*int\s*=\s*256[^\n]*",
    "max_nesting: int = 6144",
    src,
)
# batch_size: 4096 -> 256 (piu' step di gradiente su dataset piccolo)
src_new = re.sub(#faccio la stessa cosa per modificare la batch size
    r"batch_size:\s*int\s*=\s*4096[^\n]*",
    "batch_size: int = 256",
    src_new,
)

assert src_new != src, "Sostituzioni fallite: controlla manualmente config.py"
cfg_path.write_text(src_new)

# Verifica
out = subprocess.run(
    ["grep", "-nE", "(max_nesting|batch_size):", str(cfg_path)],
    capture_output=True, text=True,
).stdout
print("Linee patchate in config.py:")
print(out)

## 2. Copia degli embedding precomputati da Drive

I tre file `.npy` sono raw memmap prodotti dal notebook `6-embedding_extraction.ipynb` con split stratificato 70/20/10 per cluster:
- `places365_ViT-L~14_train_image_20475_768.npy` — 70% del subset (training)
- `places365_ViT-L~14_val_image_5850_768.npy` — 20% del subset (validation, usato per early stopping)
- `places365_ViT-L~14_test_image_2925_768.npy` — 10% del subset (test, valutato una sola volta a fine training)

Vengono copiati in `MSAE/data/` per matchare la convenzione di path della repo.

In [ ]:
import shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/tesi")
DATA  = Path("/content/MSAE/data")
DATA.mkdir(exist_ok=True)

# Cerca i file tramite glob: le dimensioni esatte dipendono dallo split stratificato
patterns = [
    "places365_ViT-L~14_train_image_*_768.npy",
    "places365_ViT-L~14_val_image_*_768.npy",
    "places365_ViT-L~14_test_image_*_768.npy",
]

for pattern in patterns:
    matches = list(DRIVE.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Nessun file corrispondente a '{pattern}' in {DRIVE}")
    src = matches[0]
    dst = DATA / src.name
    shutil.copy(src, dst)
    print(f"Copiato: {src.name}  ({dst.stat().st_size / 1e6:.1f} MB)")

## 3. Verifica GPU

Il T4 di Colab dovrebbe essere disponibile (Runtime → Change runtime type → GPU).
Il training su CPU sarebbe ~20-50× più lento.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU disponibile: {torch.cuda.get_device_name(0)}")
    print(f"VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ATTENZIONE: nessuna GPU rilevata. Cambia runtime!")

## 4. Training del MSAE

Lancia `train.py` con la configurazione **MSAE_RW** (Matryoshka SAE con reverse weighting):

- `-dt` / `-dv` / `-ds`: file di train, validation e test in formato raw memmap
- `-m MSAE_RW`: variante Matryoshka con pesatura inversa `{h, h-1, ..., 1}` , secondo il paper RW migliora la sparsity (L0 ~0.83 vs 0.75 di UW) con degrado trascurabile della reconstruction quality
- `-a TopKReLU`: attivazione che enforce sparsity tramite top-k selection (TopK rimosso a inference, resta solo ReLU)
- `--expansion_factor 8`: 768 × 8 = 6144 dimensioni latenti (come nel paper)
- `--epochs 200`: tetto massimo; l'early stopping (patience=20 epoche senza miglioramento di val_loss) interromperà prima se necessario. Il modello salvato corrisponde sempre al best checkpoint sulla validation.

Dopo il patch di `config.py` la K-list Matryoshka diventa `[64, 128, 256, 512, 1024, 2048, 4096, 6144]` (8 livelli annidati) invece del default `[64, 128, 256]`.

**Cosa monitorare nei log delle prime 5 epoche** (sanity check per decidere se proseguire):
- **CKNNA All / CKNNA Sparse** → devono **salire** (target paper ~0.99). Se scendono o restano piatte, il dataset è troppo piccolo per allenare 6144 latenti: accetta il limite o amplia il subset.
- **FVU Sparse / FVU All** → devono scendere (target paper < 0.1)
- **Sparsity (L0)** → target ~0.8 (80% dei latenti a zero)
- **Dead neurons** → idealmente vicino a zero

Il training salva un file `.pth` nella cwd a fine training.

In [ ]:
import glob

# Trova i file effettivi nella cartella data/ (nomi dipendono dalle dimensioni dello split)
train_f = glob.glob("data/places365_ViT-L~14_train_image_*_768.npy")[0]
val_f   = glob.glob("data/places365_ViT-L~14_val_image_*_768.npy")[0]
test_f  = glob.glob("data/places365_ViT-L~14_test_image_*_768.npy")[0]

print(f"Train : {train_f}")
print(f"Val   : {val_f}")
print(f"Test  : {test_f}")

!python train.py \
  -dt {train_f} \
  -dv {val_f} \
  -ds {test_f} \
  -m MSAE_RW -a TopKReLU \
  --expansion_factor 8 --epochs 200

## 5. Salvataggio del modello su Drive

Il file `.pth` viene generato in `/content/MSAE/` con un nome auto-generato che codifica gli iperparametri (es. `6144_768_TopKReLU_64_UW_False_False_0.0_places365_ViT-L~14_train_image_26325_768.pth`).

Lo copio su Drive così da poterlo riusare in fasi successive (estrazione SAE embedding, evaluation, fuzzy inference).

In [ ]:
import glob
import shutil
import pathlib

DRIVE = pathlib.Path("/content/drive/MyDrive/tesi")
MODELS_DIR = DRIVE / "models"
MODELS_DIR.mkdir(exist_ok=True)

pth_files = glob.glob("/content/MSAE/*.pth")
if not pth_files:
    raise FileNotFoundError("Nessun file .pth trovato in /content/MSAE/. Il training è andato a buon fine?")

for f in pth_files:
    src = pathlib.Path(f)
    dst = MODELS_DIR / src.name
    shutil.copy(src, dst)
    print(f"Salvato su Drive: {dst}  ({dst.stat().st_size / 1e6:.1f} MB)")